# Energy demand forecasting: feature engineering & EDA


## Setup & configuration

In [5]:
import warnings
from dataclasses import dataclass, field
from typing import Literal, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [6]:
CSV_PATH = r"C:\Actually Important Stuff\Work\test\PythonTest\EnergyOracle\data\processed\singapore\half_hourly.csv"

# Configuration object

In [7]:
@dataclass
class FeatureConfig:
    timestamp_col: str
    target_col: str

    # try to use interval-START so that the first period is 00:00-00:30 everywhere.
    timestamp_label: Literal["start", "end"] = "start"

    tz: Optional[str] = None

    freq: Optional[str] = None

    lags: tuple[str, ...] = ("30min", "1h", "1D", "7D")
    rolling_windows: tuple[str, ...] = ("3h", "1D", "7D")

    country: Optional[str] = None

    min_valid_value: float = 0.0

    cyclical: tuple[str, ...] = ("time_of_day", "day_of_week", "day_of_year")

    extra_drop: tuple[str, ...] = field(default_factory=tuple)

### Transformation: raw frame to regular, interval-start indexed series

- Parse timestamps 
- Handle timezone 
- Force target to numeric and flag non-positive values 
- Sort & remove duplicate 
- Infer sampling interval 
- Transform interval-end stamps to interval-start for parity 
- Form a complete matrix with gaps in lagging and rolling features becoming NaN rows, flagged as `is_imputed_row`.

In [8]:
def read_freq(idx: pd.DatetimeIndex) -> pd.Timedelta:
    
    step = pd.Series(idx).diff().dropna()
    if step.empty:
        raise ValueError("Not enough timestamps to infer frequency.")
    return step.mode().iloc[0]


def read_label(idx: pd.DatetimeIndex, step: pd.Timedelta) -> Optional[str]:
    
    first, last = idx[0], idx[-1]
    first_off = first - first.normalize()
    last_off = last - last.normalize()
    if first_off == step and last_off == pd.Timedelta(0):
        return "end"
    if first_off == pd.Timedelta(0):
        return "start"
    return None

def clean_series(df: pd.DataFrame, cfg: FeatureConfig) -> tuple[pd.DataFrame, dict]:
    """
    Returns a cleaned frame with interval-start timestamps and 
    'is_imputed_row' column to mark rows with NaN data
    Also returns a report describing the data
    """
    for c in (cfg.timestamp_col, cfg.target_col):
        if c not in df.columns:
            raise KeyError(f"Column '{c}' not found. Available: {list(df.columns)}")

    report: dict = {"rows_in": len(df)}
    work = df[[cfg.timestamp_col, cfg.target_col]].copy()

    # timestamps
    ts = pd.to_datetime(work[cfg.timestamp_col], errors="coerce", utc=False)
    
    # normalise via UTC.
    if ts.dtype == object:
        ts = pd.to_datetime(work[cfg.timestamp_col], errors="coerce", utc=True)
    report["unparseable_timestamps"] = int(ts.isna().sum())

    if ts.dt.tz is None:
        if cfg.tz:
            ts = ts.dt.tz_localize(cfg.tz, ambiguous="NaT", nonexistent="NaT")
    elif cfg.tz:
        ts = ts.dt.tz_convert(cfg.tz)

    work.index = pd.DatetimeIndex(ts, name="timestamp")
    work = work.drop(columns=[cfg.timestamp_col])
    work = work[work.index.notna()]

    # target column 
    work[cfg.target_col] = pd.to_numeric(work[cfg.target_col], errors="coerce")
    report["non_numeric_or_null"] = int(work[cfg.target_col].isna().sum())
    bad = work[cfg.target_col] <= cfg.min_valid_value
    report["at_or_below_min_valid"] = int(bad.sum())
    work.loc[bad, cfg.target_col] = np.nan

    # order & remove duplicates 
    if not work.index.is_monotonic_increasing:
        report["was_unsorted"] = True
        work = work.sort_index()
    dups = work.index.duplicated(keep="last")
    report["duplicate_timestamps"] = int(dups.sum())
    work = work[~dups]

    # frequency & interval label 
    step = pd.Timedelta(cfg.freq) if cfg.freq else read_freq(work.index)
    report["freq"] = str(step)

    label = read_label(work.index, step)
    report["label"] = label
    if label and label != cfg.timestamp_label:
        warnings.warn(
            f"Config timestamp_label='{cfg.timestamp_label}'and data dont match")

    if cfg.timestamp_label == "end":
        work.index = work.index - step
    work.index.name = "timestamp"

    # complete grid 
    full = pd.date_range(work.index.min(), work.index.max(), freq=step)
    n_before = len(work)
    work = work.reindex(full)
    work.index.name = "timestamp"
    report["gap_rows_inserted"] = int(len(full) - n_before)
    work["is_imputed_row"] = work[cfg.target_col].isna()
    report["missing_target_rows"] = int(work["is_imputed_row"].sum())
    report["rows_out"] = len(work)
    report["start"], report["end"] = str(work.index[0]), str(work.index[-1])
    report["span_days"] = round((work.index[-1] - work.index[0] + step) / pd.Timedelta("1D"), 2)
    return work, report

### Calendar features

In [20]:
def add_calendar_features(df: pd.DataFrame, cfg: FeatureConfig) -> pd.DataFrame:
    out = df.copy()
    idx = out.index
    out["time_of_day"] = idx.hour + idx.minute / 60.0
    out["hour"] = idx.hour
    out["day_of_week"] = idx.day_of_week                       # Monday=0
    out["is_weekend"] = (idx.day_of_week >= 5).astype(int)
    out["day_of_month"] = idx.day
    out["day_of_year"] = idx.day_of_year
    out["week_of_year"] = idx.isocalendar().week.astype(int).values
    out["month"] = idx.month
    out["quarter"] = idx.quarter
    out["year"] = idx.year

    # cyclical features for distance based regression algorithms
    period = {"time_of_day": 24.0, "day_of_week": 7.0, "day_of_year": 365.25,
              "month": 12.0}
    for col in cfg.cyclical:
        p = period[col]
        out[f"{col}_sin"] = np.sin(2 * np.pi * out[col] / p)
        out[f"{col}_cos"] = np.cos(2 * np.pi * out[col] / p)
    
    return out

### Lag and rolling features

In [10]:
def steps(duration: str, step: pd.Timedelta) -> Optional[int]:

    n = pd.Timedelta(duration) / step
    if n < 1 or n != int(n):
        warnings.warn(f"Skipping '{duration}': not a whole multiple of the {step} sampling interval.", stacklevel=3)
        return None
    return int(n)


def add_lag_features(df: pd.DataFrame, cfg: FeatureConfig) -> pd.DataFrame:

    out = df.copy()
    step = read_freq(out.index)
    y = out[cfg.target_col]
    
    for lag in cfg.lags:
        n = steps(lag, step)
        if n is None:
            continue
        if n >= len(out):
            warnings.warn(f"Skipping lag '{lag}': longer than the {len(out)}-row history.", stacklevel=2)
            continue
        out[f"lag_{lag}"] = y.shift(n)
    return out

def add_rolling_features(df: pd.DataFrame, cfg: FeatureConfig) -> pd.DataFrame:
    
    out = df.copy()
    step = read_freq(out.index)
    y_past = out[cfg.target_col].shift(1)
    
    for w in cfg.rolling_windows:
        n = steps(w, step)
        if n is None:
            continue
        if n >= len(out):
            warnings.warn(f"Skipping rolling window '{w}': longer than the {len(out)}-row history.", stacklevel=2)
            continue
        roll = y_past.rolling(n, min_periods=n)
        out[f"roll_mean_{w}"] = roll.mean()
        out[f"roll_std_{w}"] = roll.std()
        out[f"roll_max_{w}"] = roll.max()
        out[f"roll_min_{w}"] = roll.min()
    
    return out

def build_features(clean: pd.DataFrame, 
                   cfg: FeatureConfig,
                   include_lags: bool = True, 
                   include_rolling: bool = True) -> pd.DataFrame:
    """clean_series() output -> modelling frame. Target column is kept as-is."""
    
    out = add_calendar_features(clean, cfg)
    if include_lags:
        out = add_lag_features(out, cfg)
    if include_rolling:
        out = add_rolling_features(out, cfg)
    return out.drop(columns=list(cfg.extra_drop), errors="ignore")

### Chronological train-test split
To avoid data leakage from a random train-test split the data is split chronologically. 

In [11]:
def chronological_split(df: pd.DataFrame, train_frac: float = 0.7, val_frac: float = 0.15):
    n = len(df)
    i, j = int(n * train_frac), int(n * (train_frac + val_frac))
    return df.iloc[:i], df.iloc[i:j], df.iloc[j:]

# Apply to data

In [12]:
raw = pd.read_csv(CSV_PATH)
print(raw.shape)
raw.head(10)

(69888, 7)


,ts_end_singaporetime,date,period,system_demand_mw,nem_forecast_mw,source_file,file_hash
0,2022-11-07 00:30:00+08:00,2022-11-07,1,5676.678890,5574.5,20221107.xls,44d30eda275459a94b38aa4f32b608dbae35b92fa7d835...
1,2022-11-07 01:00:00+08:00,2022-11-07,2,5556.767025,5462.8,20221107.xls,44d30eda275459a94b38aa4f32b608dbae35b92fa7d835...
2,2022-11-07 01:30:00+08:00,2022-11-07,3,5475.695596,5436.7,20221107.xls,44d30eda275459a94b38aa4f32b608dbae35b92fa7d835...
3,2022-11-07 02:00:00+08:00,2022-11-07,4,5410.115955,5365.2,20221107.xls,44d30eda275459a94b38aa4f32b608dbae35b92fa7d835...
4,2022-11-07 02:30:00+08:00,2022-11-07,5,5358.712663,5296.7,20221107.xls,44d30eda275459a94b38aa4f32b608dbae35b92fa7d835...
5,2022-11-07 03:00:00+08:00,2022-11-07,6,5316.136343,5283.5,20221107.xls,44d30eda275459a94b38aa4f32b608dbae35b92fa7d835...
6,2022-11-07 03:30:00+08:00,2022-11-07,7,5293.564363,5240.0,20221107.xls,44d30eda275459a94b38aa4f32b608dbae35b92fa7d835...
7,2022-11-07 04:00:00+08:00,2022-11-07,8,5270.425313,5207.6,20221107.xls,44d30eda275459a94b38aa4f32b608dbae35b92fa7d835...
8,2022-11-07 04:30:00+08:00,2022-11-07,9,5265.957225,5227.6,20221107.xls,44d30eda275459a94b38aa4f32b608dbae35b92fa7d835...
9,2022-11-07 05:00:00+08:00,2022-11-07,10,5277.711423,5207.1,20221107.xls,44d30eda275459a94b38aa4f32b608dbae35b92fa7d835...


In [13]:
cfg = FeatureConfig(
    timestamp_col="ts_end_singaporetime",
    target_col="system_demand_mw",
    timestamp_label="end",
    country="SG",
)

clean, report = clean_series(raw, cfg)
y = clean[cfg.target_col]
pd.Series(report, name="value").to_frame()

,value
rows_in,69888
unparseable_timestamps,0
non_numeric_or_null,0
at_or_below_min_valid,0
duplicate_timestamps,0
freq,0 days 00:30:00
label,end
gap_rows_inserted,0
missing_target_rows,0
rows_out,69888


In [21]:
feats = build_features(clean, cfg)
print(f"{feats.shape[0]} rows x {feats.shape[1]} columns")
feats.head()

69888 rows x 34 columns


,system_demand_mw,is_imputed_row,time_of_day,hour,day_of_week,is_weekend,day_of_month,day_of_year,week_of_year,month,...,roll_max_3h,roll_min_3h,roll_mean_1D,roll_std_1D,roll_max_1D,roll_min_1D,roll_mean_7D,roll_std_7D,roll_max_7D,roll_min_7D
timestamp,,,,,,,,,,,,,,,,,,,,,
2022-11-07 00:00:00+08:00,5676.678890,False,0.0,0,0,0,7,311,45,11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-11-07 00:30:00+08:00,5556.767025,False,0.5,0,0,0,7,311,45,11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-11-07 01:00:00+08:00,5475.695596,False,1.0,1,0,0,7,311,45,11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-11-07 01:30:00+08:00,5410.115955,False,1.5,1,0,0,7,311,45,11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-11-07 02:00:00+08:00,5358.712663,False,2.0,2,0,0,7,311,45,11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
feats.columns

Index(['system_demand_mw', 'is_imputed_row', 'time_of_day', 'hour',
       'day_of_week', 'is_weekend', 'day_of_month', 'day_of_year',
       'week_of_year', 'month', 'quarter', 'year', 'time_of_day_sin',
       'time_of_day_cos', 'day_of_week_sin', 'day_of_week_cos',
       'day_of_year_sin', 'day_of_year_cos', 'lag_30min', 'lag_1h', 'lag_1D',
       'lag_7D', 'roll_mean_3h', 'roll_std_3h', 'roll_max_3h', 'roll_min_3h',
       'roll_mean_1D', 'roll_std_1D', 'roll_max_1D', 'roll_min_1D',
       'roll_mean_7D', 'roll_std_7D', 'roll_max_7D', 'roll_min_7D'],
      dtype='str')

In [ ]:
# These new features introduce some NaN values into the dataframe 
# but the amount of rows affected by NaN values is less than 0.5% of the data
nan_counts = feats.isna().sum()
nan_counts[nan_counts > 0]

lag_30min         1
lag_1h            2
lag_1D           48
lag_7D          336
roll_mean_3h      6
roll_std_3h       6
roll_max_3h       6
roll_min_3h       6
roll_mean_1D     48
roll_std_1D      48
roll_max_1D      48
roll_min_1D      48
roll_mean_7D    336
roll_std_7D     336
roll_max_7D     336
roll_min_7D     336
dtype: int64